# Callbacks in Agentic Systems — LangGraph + OpenAI Hands-On Notebook

**Companion notebook to the tutorial doc:** *Callbacks in AI Agents — A LangGraph + OpenAI Tutorial*

This notebook walks you through building a fully **observable agent** using LangGraph's state-machine
orchestration, OpenAI tool-calling models, and a custom `BaseCallbackHandler` that captures every
lifecycle event: LLM start/end, token usage, tool start/end, errors, and agent actions.

> **Prerequisites:** Python 3.10+, an OpenAI API key (`OPENAI_API_KEY`).

## 1. Installation & Setup

Install the required packages and configure your OpenAI API key.

In [ ]:
# # Run once — installs LangChain / LangGraph / OpenAI integration
# !pip install -q langchain langchain-openai langgraph langchain-core

In [1]:
import os
import getpass

# Set your OpenAI API key securely (prompts if not already in the environment)
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

print("API key configured.")

API key configured.


## 2. Why Callbacks?

Callbacks are **hooks into the agent's execution lifecycle**. In production agentic systems they power:

| Use case | What the callback captures |
|---|---|
| **Observability** | Which node/tool ran, in what order, with what input |
| **Cost control** | Token usage per LLM call (via `on_llm_end`) |
| **Debugging** | Exact prompts sent and raw outputs returned |
| **Streaming UX** | Token-by-token output (`on_llm_new_token`) |
| **Audit / compliance** | Timestamped log of every model + tool invocation |
| **Resilience** | Error interception (`on_llm_error`, `on_tool_error`) |

The handler below implements all of the core events.

## 3. The Custom Callback Handler

`AgentCallbackHandler` subclasses `BaseCallbackHandler` and records structured, timestamped
log entries for every lifecycle event, plus running counters for LLM calls, tool calls, and tokens.

In [2]:
from langchain_core.callbacks import BaseCallbackHandler
from datetime import datetime, timezone


class AgentCallbackHandler(BaseCallbackHandler):
    """Comprehensive callback handler for agent observability."""

    def __init__(self):
        self.llm_calls = 0
        self.tool_calls = 0
        self.total_tokens = 0
        self.logs = []

    # ---------- LLM lifecycle ----------
    def on_llm_start(self, serialized, prompts, **kwargs):
        self.llm_calls += 1
        entry = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "event": "llm_start",
            "call_number": self.llm_calls,
            "prompt_preview": str(prompts)[:120],
        }
        self.logs.append(entry)
        print(f"\n[LLM START]  call #{self.llm_calls}")

    def on_llm_end(self, response, **kwargs):
        tokens = 0
        if getattr(response, "llm_output", None):
            tokens = response.llm_output.get("token_usage", {}).get("total_tokens", 0)
        self.total_tokens += tokens
        self.logs.append({
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "event": "llm_end",
            "tokens_used": tokens,
            "cumulative_tokens": self.total_tokens,
        })
        print(f"[LLM END]    tokens={tokens} | cumulative={self.total_tokens}")

    def on_llm_error(self, error, **kwargs):
        self.logs.append({
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "event": "llm_error",
            "error": str(error),
        })
        print(f"[LLM ERROR]  {error}")

    # ---------- Tool lifecycle ----------
    def on_tool_start(self, serialized, input_str, **kwargs):
        self.tool_calls += 1
        tool_name = serialized.get("name", "unknown")
        self.logs.append({
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "event": "tool_start",
            "tool_name": tool_name,
            "input_preview": str(input_str)[:120],
        })
        print(f"\n[TOOL START] {tool_name}  input={str(input_str)[:80]}...")

    def on_tool_end(self, output, **kwargs):
        self.logs.append({
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "event": "tool_end",
            "output_preview": str(output)[:120],
        })
        print(f"[TOOL END]   output={str(output)[:80]}...")

    def on_tool_error(self, error, **kwargs):
        self.logs.append({
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "event": "tool_error",
            "error": str(error),
        })
        print(f"[TOOL ERROR] {error}")

    # ---------- Agent lifecycle ----------
    def on_agent_action(self, action, **kwargs):
        self.logs.append({
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "event": "agent_action",
            "tool": action.tool,
            "tool_input_preview": str(action.tool_input)[:120],
        })
        print(f"\n[AGENT ACTION] -> {action.tool}")

    def on_agent_finish(self, finish, **kwargs):
        self.logs.append({
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "event": "agent_finish",
            "output_preview": str(finish.return_values)[:120],
        })
        print("\n[AGENT FINISH] task complete")

    # ---------- Reporting ----------
    def get_summary(self):
        return {
            "total_llm_calls": self.llm_calls,
            "total_tool_calls": self.tool_calls,
            "total_tokens": self.total_tokens,
            "execution_logs": self.logs,
        }

## 4. Define Tools

Three simple tools the agent can call. In a real system these would wrap APIs, databases,
or internal services — callbacks make each invocation visible.

In [3]:
from langchain_core.tools import tool


@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location."""
    weather_data = {
        "New York": "Sunny, 72F",
        "London": "Rainy, 15C",
        "Tokyo": "Cloudy, 20C",
        "Mumbai": "Hot and humid, 32C",
    }
    return weather_data.get(location, f"Weather data not available for {location}")


@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression, e.g. '25 * 4 + 10'."""
    try:
        return f"Result: {eval(expression)}"
    except Exception as e:
        return f"Error calculating: {e}"


@tool
def search_database(query: str) -> str:
    """Search a knowledge base for information about data/AI technologies."""
    results = {
        "databricks": "Databricks is a unified analytics platform built on Apache Spark.",
        "spark": "Apache Spark is a distributed computing framework for big data processing.",
        "mlflow": "MLflow is an open-source platform for managing the ML lifecycle.",
    }
    for key, value in results.items():
        if key in query.lower():
            return value
    return "No results found in database"


tools = [get_weather, calculate, search_database]
print(f"Registered {len(tools)} tools: {[t.name for t in tools]}")

Registered 3 tools: ['get_weather', 'calculate', 'search_database']


## 5. Build the LangGraph Agent

The agent is a **state machine** with two nodes:

- `agent` — calls the OpenAI model (with tools bound); the model either answers or requests tool calls
- `tools` — `ToolNode` executes the requested tool calls and appends results to state

A conditional edge loops `agent -> tools -> agent` until the model produces a final answer
with no tool calls.

**Wiring the callback in two places:**
1. On the LLM constructor (`callbacks=[handler]`) — captures model-level events.
2. On `app.invoke(..., config={"callbacks": [handler]})` — propagates to the full run,
   including `ToolNode` executions.

In [4]:
from typing import TypedDict, Annotated, Sequence
import operator

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode


class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]


def build_agent_graph(callback_handler):
    """Build the two-node agent graph with callbacks wired in."""

    # 1) LLM with callback attached at the model level
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
        callbacks=[callback_handler],
    )
    llm_with_tools = llm.bind_tools(tools)

    # 2) Agent node: model reasoning step
    def agent_node(state: AgentState) -> AgentState:
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}

    # 3) Router: continue to tools, or finish
    def should_continue(state: AgentState) -> str:
        last = state["messages"][-1]
        if getattr(last, "tool_calls", None):
            return "tools"
        return "end"

    workflow = StateGraph(AgentState)
    workflow.add_node("agent", agent_node)
    workflow.add_node("tools", ToolNode(tools))
    workflow.set_entry_point("agent")
    workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
    workflow.add_edge("tools", "agent")

    return workflow.compile()


print("Graph builder defined.")

Graph builder defined.


## 6. Run the Agent (with full callback trace)

`run_agent_with_callbacks` creates a fresh handler per run, executes the graph, prints the
live trace, the final answer, and a summary with token totals and the structured event log.

In [5]:
import json
from langchain_core.messages import HumanMessage


def run_agent_with_callbacks(user_input: str):
    print("=" * 70)
    print(f"USER INPUT: {user_input}")
    print("=" * 70)

    handler = AgentCallbackHandler()
    app = build_agent_graph(handler)

    final_state = app.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config={"callbacks": [handler]},   # run-level: propagates into ToolNode
    )

    final_message = final_state["messages"][-1]
    print("\n" + "=" * 70)
    print("FINAL RESPONSE:")
    print("=" * 70)
    print(final_message.content)

    summary = handler.get_summary()
    print("\n" + "=" * 70)
    print("EXECUTION SUMMARY")
    print("=" * 70)
    print(f"LLM calls : {summary['total_llm_calls']}")
    print(f"Tool calls: {summary['total_tool_calls']}")
    print(f"Tokens    : {summary['total_tokens']}")
    return final_state, summary

## 7. Examples

### Example 1 — Single-tool query (weather + calculation)

In [6]:
state1, summary1 = run_agent_with_callbacks(
    "What's the weather in Tokyo, and convert 20C to Fahrenheit?"
)

USER INPUT: What's the weather in Tokyo, and convert 20C to Fahrenheit?

[LLM START]  call #1
[LLM END]    tokens=168 | cumulative=168

[TOOL START] get_weather  input={'location': 'Tokyo'}...
[TOOL END]   output=content='Cloudy, 20C' name='get_weather' tool_call_id='call_ptFW5Q54aGAfcuplgm7v...

[TOOL START] calculate  input={'expression': '(20 * 9/5) + 32'}...
[TOOL END]   output=content='Result: 68.0' name='calculate' tool_call_id='call_X1PuGbqdhiH2lahbE61Me...

[LLM START]  call #2
[LLM END]    tokens=223 | cumulative=391

FINAL RESPONSE:
The weather in Tokyo is currently cloudy with a temperature of 20°C. In Fahrenheit, 20°C is equivalent to 68°F.

EXECUTION SUMMARY
LLM calls : 2
Tool calls: 2
Tokens    : 391


### Example 2 — Knowledge-base lookup with reasoning

In [7]:
state2, summary2 = run_agent_with_callbacks(
    "Search the database for Databricks and explain in one sentence why it is useful."
)

USER INPUT: Search the database for Databricks and explain in one sentence why it is useful.

[LLM START]  call #1
[LLM END]    tokens=134 | cumulative=134

[TOOL START] search_database  input={'query': 'Databricks'}...
[TOOL END]   output=content='Databricks is a unified analytics platform built on Apache Spark.' name...

[LLM START]  call #2
[LLM END]    tokens=192 | cumulative=326

FINAL RESPONSE:
Databricks is useful because it provides a unified analytics platform built on Apache Spark, enabling data teams to collaborate and streamline the process of data engineering, machine learning, and analytics.

EXECUTION SUMMARY
LLM calls : 2
Tool calls: 1
Tokens    : 326


### Example 3 — Multi-tool task (forces several tool calls + multiple LLM turns)

In [8]:
state3, summary3 = run_agent_with_callbacks(
    "Get the weather for Mumbai, search the database for Spark, and calculate 25 * 4 + 10."
)

USER INPUT: Get the weather for Mumbai, search the database for Spark, and calculate 25 * 4 + 10.

[LLM START]  call #1
[LLM END]    tokens=187 | cumulative=187

[TOOL START] get_weather  input={'location': 'Mumbai'}...
[TOOL END]   output=content='Hot and humid, 32C' name='get_weather' tool_call_id='call_u5c8Yy7Eo2bzI...

[TOOL START] search_database  input={'query': 'Spark'}...
[TOOL END]   output=content='Apache Spark is a distributed computing framework for big data processi...

[TOOL START] calculate  input={'expression': '25 * 4 + 10'}...
[TOOL END]   output=content='Result: 110' name='calculate' tool_call_id='call_PndhHYSPqXsYl6Hv12HtIa...

[LLM START]  call #2
[LLM END]    tokens=286 | cumulative=473

FINAL RESPONSE:
Here are the results for your requests:

- **Weather in Mumbai**: Hot and humid, 32°C.
- **Information about Spark**: Apache Spark is a distributed computing framework for big data processing.
- **Calculation Result**: 25 * 4 + 10 = 110.

EXECUTION SUMMARY
LLM call

### Inspect the structured event log

Every callback event was captured as a JSON-serializable dict — ready to ship to a log store
(Elastic, MongoDB, Datadog) or an observability platform like LangSmith.

In [10]:
print(json.dumps(summary3["execution_logs"], indent=2))

[
  {
    "timestamp": "2026-09-07T05:56:52.825626+00:00",
    "event": "llm_start",
    "call_number": 1,
    "prompt_preview": "['Human: Get the weather for Mumbai, search the database for Spark, and calculate 25 * 4 + 10.']"
  },
  {
    "timestamp": "2026-09-07T05:56:54.262798+00:00",
    "event": "llm_end",
    "tokens_used": 187,
    "cumulative_tokens": 187
  },
  {
    "timestamp": "2026-09-07T05:56:54.267808+00:00",
    "event": "tool_start",
    "tool_name": "get_weather",
    "input_preview": "{'location': 'Mumbai'}"
  },
  {
    "timestamp": "2026-09-07T05:56:54.268807+00:00",
    "event": "tool_end",
    "output_preview": "content='Hot and humid, 32C' name='get_weather' tool_call_id='call_u5c8Yy7Eo2bzIHVgIT0r9SbW'"
  },
  {
    "timestamp": "2026-09-07T05:56:54.269811+00:00",
    "event": "tool_start",
    "tool_name": "search_database",
    "input_preview": "{'query': 'Spark'}"
  },
  {
    "timestamp": "2026-09-07T05:56:54.270809+00:00",
    "event": "tool_end",
    "out

## 8. Bonus — Streaming Callback (token-by-token)

For chat UIs, implement `on_llm_new_token` to stream tokens to the client as they are generated.
Note: streaming requires `streaming=True` on the model.

In [12]:
class StreamingCallbackHandler(BaseCallbackHandler):
    """Print tokens as they are generated (pipe to websocket/SSE in a real app)."""

    def on_llm_new_token(self, token: str, **kwargs):
        print(token, end="", flush=True)


streaming_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    streaming=True,
    callbacks=[StreamingCallbackHandler()],
)

print("Streaming reply: ", end="")
_ = streaming_llm.invoke("In one short sentence, what is a callback in an AI agent?")
print()

Streaming reply: A callback in an AI agent is a function that is passed as an argument to another function and is executed after a specific event or condition is met.


## 9. Exercises

1. **Latency tracking** — extend the handler to record `time.perf_counter()` in `on_llm_start`
   and compute per-call latency in `on_llm_end`.
2. **Cost estimator** — given `gpt-4o-mini` pricing, compute per-run cost from the token usage
   (hint: `token_usage` also has `prompt_tokens` and `completion_tokens`).
3. **Error injection** — make `calculate` raise on division by zero and verify `on_tool_error`
   fires; then make the agent recover gracefully.
4. **Async logging** — convert the handler to `AsyncCallbackHandler` and `await` a (mocked)
   write to an external logging service.
5. **PII redaction** — redact email-like patterns from prompts before logging them.

---
*End of notebook. See the companion tutorial doc for the full conceptual walkthrough.*